In [ ]:
%pip install peft evaluate

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
except:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        print(e)

In [ ]:
from huggingface_hub import sync_bucket

sync_bucket(
    "hf://buckets/RobChio/swissgerman",
    "./bucket/"
)

In [ ]:
labels = {
    0: "AG",
    1: "BE",
    2: "BS",
    3: "GR",
    4: "LU",
    5: "SG",
    6: "VS",
    7: "ZH",
}

id2label = labels
label2id = {v: k for k, v in labels.items()}
num_labels = len(labels)

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperForAudioClassification, WhisperProcessor
import torch

model_id = "Flix-AI/flix-swissgerman-full"

processor = WhisperProcessor.from_pretrained(model_id)

model = WhisperForAudioClassification.from_pretrained(
    model_id,
    num_labels=8,
    label2id=label2id,
    id2label=id2label,
    torch_dtype=torch.bfloat16,
)

# SpecAugment
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.mask_feature_prob = 0.05

In [ ]:
from peft import PeftModel, LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    # target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0.05,
    bias="none",
    modules_to_save=["projector", "classifier"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
import datasets
from datasets import load_dataset, load_from_disk, Audio, interleave_datasets

ds_swissdial = load_dataset("RobChio/swiss-dial-preprocessed", split="train", streaming=False) # enable streaming!
ds_swissdial.set_format(type="torch", columns=["input_features", "labels"])

ds_city = load_from_disk("./bucket/preprocessed_swiss_train")
ds_city_eval = load_from_disk("./bucket/preprocessed_swiss_eval")
ds_city.set_format("torch")
ds_city_eval.set_format("torch")

dataset = interleave_datasets(
    [ds_swissdial, ds_city],
    probabilities=[0.75, 0.25],
    seed=42
)

train_dataset = dataset.shuffle(seed=42)
val_dataset = ds_city_eval

# train_val_split = dataset.train_test_split(test_size=0.1)
# train_dataset = train_val_split["train"]
# val_dataset = train_val_split["test"]

print(len(dataset))

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np
import torch.nn.functional as F

class FastFeatureCollator:
    def __call__(self, features):
        input_features = torch.stack([f["input_features"] for f in features]).to(torch.bfloat16)
        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return {"input_features": input_features, "labels": labels}

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)
    
    macro_f1 = f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
    acc = accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    return {"f1": macro_f1, "accuracy": acc}

training_args = TrainingArguments(
    output_dir="./swissgerman-dialect-classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    #max_steps=120, # for SwissDial streaming 11213 / 256 * 3 epochs ## its actually 27828 and 3093
    num_train_epochs=3,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1", #"accuracy"
    greater_is_better=True,
    bf16=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0, # multiple workers with streaming causes data duplication
    torch_compile=False, #True,
    dataloader_drop_last=False,
    push_to_hub=True,
    hub_model_id="RobChio/swissgerman-dialect-classifier",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=FastFeatureCollator(),
    compute_metrics=compute_metrics,
)

In [ ]:
# Train model
train_result = trainer.train()

# Log and save training state and metrics
# trainer.log_metrics("train", train_result.metrics)
# trainer.save_metrics("train", train_result.metrics)
# trainer.save_state()

In [ ]:
trainer.push_to_hub(commit_message="Finished training")

In [ ]:
# eval_metrics = trainer.evaluate(eval_dataset=test_dataset)
# trainer.log_metrics("eval", eval_metrics)
# trainer.save_metrics("eval", eval_metrics)

In [ ]:
# from google.colab import runtime
# runtime.unassign()